# Loss landscape over tongueIndex

Sanity check that gradient descent through the synth can work: for each of several reference
configurations, render 1 s of reference audio, then sweep `tongueIndex` across its range
(everything else held at the reference values) and plot the MFCC loss against the reference.

Same setup as training: `pink_trombone_ola` at 512 samples per frame with 256-sample impulse
responses, `MFCCLoss(n_fft=2048)`. The noise seed is shared between the reference and every
candidate, so the loss is exactly 0 at the reference value.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import plotly.graph_objects as go
import torch
from IPython.display import Audio, display

from samuel.losses import MFCCLoss
from samuel.model import _DEFAULT_FROZEN_VALUES, _DEFAULT_PARAM_SPEC
from samuel.pink_trombone import PARAM_NAMES, SAMPLE_RATE, pink_trombone_ola

SAMPLES_PER_FRAME = 512
CONTROL_RATE = SAMPLE_RATE / SAMPLES_PER_FRAME
N_FRAMES = round(1.0 * CONTROL_RATE)  # 1 s
SEED = 0
TI = PARAM_NAMES.index("tongueIndex")
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# The loss stays on CPU: torch.stft on MPS crashed the kernel. It's cheap next to the synth.
loss_fn = MFCCLoss(samples_per_frame=SAMPLES_PER_FRAME, n_fft=2048)

## Reference configurations

Config 0 is the model's init values with a 120 Hz pitch. The rest are drawn with a seeded
generator, uniformly within the trainable ranges, except that the tongue, constriction and lips
stay open (no closures or fricatives) and voicing stays audible, so every reference is a vowel.

In [ ]:
N_CONFIGS = 10

# Overrides of the trainable ranges for the random configs.
SAMPLE_RANGES = {
    **{name: (lo, hi) for name, (lo, hi, _) in _DEFAULT_PARAM_SPEC.items()},
    "frequency": (80.0, 220.0),
    "voiceness": (0.5, 1.0),
    "intensity": (0.5, 1.0),
    # Below ~1.6 the tongue nearly closes the tract and the output is near-silent.
    "tongueDiameter": (1.8, 3.5),
    "constrictionDiameter": (0.7, 3.0),
    "lipDiameter": (0.7, 3.0),
}

configs = [
    {
        **{name: init for name, (_, _, init) in _DEFAULT_PARAM_SPEC.items()},
        **_DEFAULT_FROZEN_VALUES,
        "frequency": 120.0,
        "tongueIndex": 25.0,
    }
]
gen = torch.Generator().manual_seed(SEED)
for _ in range(N_CONFIGS - 1):
    u = torch.rand(len(SAMPLE_RANGES), generator=gen).tolist()
    configs.append(
        {
            **_DEFAULT_FROZEN_VALUES,
            **{
                name: lo + (hi - lo) * ui
                for (name, (lo, hi)), ui in zip(SAMPLE_RANGES.items(), u)
            },
        }
    )

for i, c in enumerate(configs):
    print(i, {k: round(v, 2) for k, v in c.items() if k not in _DEFAULT_FROZEN_VALUES})

In [ ]:
def make_params(config: dict[str, float], tongue_index: torch.Tensor) -> torch.Tensor:
    """[B] tongueIndex values -> [B, N_FRAMES, 12], constant over time."""
    base = torch.tensor([config[n] for n in PARAM_NAMES], device=DEVICE)
    params = base.expand(len(tongue_index), N_FRAMES, -1).clone()
    params[..., TI] = tongue_index[:, None]
    return params


def synth(params: torch.Tensor) -> torch.Tensor:
    # A [B] tensor of equal seeds gives every row the same noise; a scalar seed
    # would draw a different noise row per batch slot.
    seed = torch.full((len(params),), SEED)
    return pink_trombone_ola(params, seed=seed, control_rate=CONTROL_RATE).cpu()


with torch.no_grad():
    ref_audios = [
        synth(make_params(c, torch.tensor([c["tongueIndex"]], device=DEVICE)))
        for c in configs
    ]

## Sweep

In [ ]:
lo, hi, _ = _DEFAULT_PARAM_SPEC["tongueIndex"]
grid = torch.linspace(lo, hi, 51, device=DEVICE)

losses = []
with torch.no_grad():
    for c, ref in zip(configs, ref_audios):
        audio = torch.cat([synth(make_params(c, chunk)) for chunk in grid.split(8)])
        losses.append(torch.stack([loss_fn(a[None], ref) for a in audio]))
grid = grid.cpu()

## Results

In [ ]:
for i, (c, ref, loss) in enumerate(zip(configs, ref_audios, losses)):
    items = [f"{k}={v:.2f}" for k, v in c.items() if k not in _DEFAULT_FROZEN_VALUES]
    desc = "<br>".join(", ".join(items[j : j + 4]) for j in range(0, len(items), 4))
    display(Audio(ref[0].numpy(), rate=SAMPLE_RATE))
    fig = go.Figure(go.Scatter(x=grid, y=loss))
    fig.add_vline(x=c["tongueIndex"], line_dash="dash", line_color="black")
    fig.update_layout(
        height=340,
        margin=dict(t=100, b=40),
        title=dict(text=f"Config {i}<br><sup>{desc}</sup>"),
        xaxis_title="tongueIndex",
        yaxis_title="MFCC loss",
    )
    fig.show()